In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


# PGAE-tau Detection / 检测评估

Set `TARGET_CASE` to `case14`, `case30`, or `case57`, then select **Restart Kernel & Run All**. Configuration is loaded when project modules are imported.

在下方设置 `TARGET_CASE`，选择 **Restart Kernel & Run All**。项目模块在导入时读取系统配置。


In [ ]:
import os

# Evaluation case: 'case14', 'case30', or 'case57'.
TARGET_CASE = os.environ.get('PGAE_CASE', 'case30')

os.environ['PGAE_CASE'] = TARGET_CASE
print(f'目标 Case: {TARGET_CASE}')
print('配置在模块导入时生效；切换系统后选择 Restart Kernel & Run All。')


In [ ]:
import matplotlib.pyplot as plt
from configs.config import sys_config
from configs.nn_setting import nn_setting
from tqdm import tqdm
from matplotlib import cm

In [ ]:
import torch
import sys

from data.loaders import load_case, load_measurement, load_load_pv, load_dataset

from models.baselines.piconvae import PIConvAE
from models.ablation.pgae_lstm import PG_AE_LSTM
from models.detection.pgae_tau import PGAE_tau



from data.dataset import rnn_dataset

from torch.utils.data import DataLoader
import numpy as np
from torch import optim
from data.dataset import scaler
from time import time

In [ ]:
class Evaluation:
    def __init__(self, case_class):

        self.nn_setting = nn_setting
        self.case_class = case_class
        self.case_class.gen_torch_constant()                  # Convert the numpy array into torch array for recovery purpose

        # Load the model and trained parameters
        model = PGAE_tau().to(nn_setting['device'])
        model_path = f"saved_model/{sys_config['case_name']}/PGAE_tau.pt"

        model.load_state_dict(torch.load(model_path, map_location=torch.device(nn_setting['device']),  weights_only=True))
        
        device = nn_setting['device']
        print(f"Using {device} device")
        model.to(device)
        model.eval()
        print(model)

        self.model = model
        self.device = device

        # Scaling factor and DeScaling factor
        self.scaler_ = scaler()
        ## numpy data
        self.min = torch.from_numpy(self.scaler_.min).float().to(nn_setting['device']).requires_grad_(False)
        self.max = torch.from_numpy(self.scaler_.max).float().to(nn_setting['device']).requires_grad_(False)
        ## tensor data
        self.t_min = torch.tensor(self.scaler_.min, device=device, dtype=torch.float32)
        self.t_max = torch.tensor(self.scaler_.max, device=device, dtype=torch.float32)
        self.t_diff = self.t_max - self.t_min + 1e-7
        
        # Load Validation dataset
        # Valid set
        valid_set = rnn_dataset(mode = 'valid', istransform = True)
        valid_loader = DataLoader(valid_set, batch_size=1, shuffle=False)
        
        # Calculate validation loss
        loss_fn = torch.nn.MSELoss()
        valid_loss = []
        with torch.no_grad():
            for input, output in valid_loader:
                assert np.all(input.numpy() == output.numpy())
                input, output = input.to(device), output.to(device)
                decoded, _ , _ = model(input)
                decoded_norm = (decoded- self.t_min)/(self.t_max - self.t_min + 1e-7)
                loss = loss_fn(decoded_norm, output)
                valid_loss.append(loss.item())

        # detection threshold
        quantile = [0.80,0.82,0.84,0.86,0.88,0.90,0.92,0.94,0.96,0.98,0.999] ### 阈值
        ae_threshold = np.quantile(valid_loss, quantile)
        self.ae_threshold = ae_threshold
        self.quantile_idx = 6   # The index of threshold quantile in evaluation
        
        print(f'The detection threshold: {self.ae_threshold}')

        self.quantile = quantile
        
    
    def evaluate(self, test_data):

        if torch.is_tensor(test_data):
            test_data = test_data.float().to(self.device)
        else:
            test_data = torch.tensor(test_data, dtype=torch.float32, device=self.device)    # convert to tensor
            
        assert test_data.shape == (1, self.nn_setting['sample_length'], self.nn_setting['feature_size'])
            
        dim = self.nn_setting['lattent_dim'] ## zeros_size 和 lattent_dim的值有关

        loss_fn = torch.nn.MSELoss()
        self.model.eval()
        with torch.no_grad():
            decoded,_,_ = self.model(test_data)
            decoded_norm = (decoded - self.t_min)/(self.t_max - self.t_min + 1e-7)
            ## data_descale = test_data * self.t_diff + self.t_min
            loss_recons = loss_fn(decoded_norm, test_data)

        
        return decoded.cpu().numpy(), loss_recons.item()
    


In [ ]:
"""
Test the performance of attack detection and identification algorithms
"""

from configs.config import sys_config, save_metric
from data.loaders import load_case,load_measurement,load_load_pv

# Load cases, measurement, and load
case_class = load_case()
z_noise_summary, v_est_summary = load_measurement()
load_active, load_reactive, pv_active_, pv_reactive_ = load_load_pv()
test_dataloader_scaled, test_dataloader_unscaled, valid_dataloader_scaled, valid_dataloader_unscaled = load_dataset()


dd_detector = Evaluation(case_class=case_class)  # Instance the data-driven detector
scaler_ = scaler()                               # Instance the scaler class
print(f'Quantile: {dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'Threshold: {dd_detector.ae_threshold[dd_detector.quantile_idx]}')

"""
Metrics
"""
# Attack list
ang_no_list = [1,2,3]
mag_no = 0
ang_str_list = [0.2,0.3] # Attack strength list
mag_str = 0

# Attack detection
TP_DDD = {}                   # Record the true positive number of deep learning detector
FP_DDD = []

# Attack recovery
att_strength = {}
recover_deviation = {}        # The difference between recovered state phase angle and the ground truth state phase angle
recovery_ite_no = {}          # The number of iterations in recovery algorithm
recovery_time = []            # The recovery time
residual_bdd = {}         # The bdd residual of the recovered measurement
residual_ddd = {}

"""
FPR of DDD
"""
print('FPR of DDD.')

for idx_, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):  
    """
    idx_: starts from 0
    idx: the actual index number in test_dataloader_unscaled
    """
    # Scale
    z_att_noise_scale = scaler_(input)
    decoded, loss_recons = dd_detector.evaluate(z_att_noise_scale)
    
    if loss_recons <= dd_detector.ae_threshold[dd_detector.quantile_idx]:
        FP_DDD.append(False)
    else:
        FP_DDD.append(True)
    
"""
TPR of DDD
"""
print('TPR of DDD.')

for ang_no in ang_no_list:
    for ang_str in ang_str_list:
        print(f'({ang_no},{ang_str})')
        """
        Construct the dict keys
        """
        TP_DDD[f'({ang_no},{ang_str})'] = []
        att_strength[f'({ang_no},{ang_str})'] = []
        
        for idx_, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):  
            """
            idx_: starts from 0
            idx: the actual index number in test_dataloader_unscaled
            """
            
            v_est_pre = v_est_pre.flatten()
            v_est_last = v_est_last.flatten()  # The ground truth state

            """
            Generate attack
            """
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(z_noise=input, v_est_last=v_est_last, ang_no=ang_no, mag_no=mag_no, ang_str=ang_str, mag_str=mag_str)
            v_att_est_last = torch.from_numpy(v_att_est_last)

            """
            Data-driven detector
            """
            # Scale
            z_att_noise_scale = scaler_(z_att_noise)
            decoded, loss_recons = dd_detector.evaluate(z_att_noise_scale)
            
            if loss_recons <= dd_detector.ae_threshold[dd_detector.quantile_idx]:
                TP_DDD[f'({ang_no},{ang_str})'].append(False)
            else:
                TP_DDD[f'({ang_no},{ang_str})'].append(True)

            # Attack strength
            vang_true = np.angle(v_est_last.numpy())
            vang_att = np.angle(v_att_est_last.numpy())
            c_true = (vang_att - vang_true)
            att_strength[f'({ang_no},{ang_str})'].append(np.linalg.norm(c_true,2))
            


save_metric(address = f'metric/{sys_config["case_name"]}/metric_ddd_test.npy', 
            # Data-driven Detector
            TP_DDD = TP_DDD,
            att_strength = att_strength,
            FP_DDD = FP_DDD,
            recover_deviation = recover_deviation,
            residual_bdd = residual_bdd,
            residual_ddd = residual_ddd
            )

In [ ]:
# Load cases, measurement, and load
case_class = load_case()

dd_detector = Evaluation(case_class=case_class)  # Instance the data-driven detector
scaler_ = scaler()     

z_noise_summary, v_est_summary = load_measurement()
load_active, load_reactive, pv_active_, pv_reactive_ = load_load_pv()
test_dataloader_scaled, test_dataloader_unscaled, valid_dataloader_scaled, valid_dataloader_unscaled = load_dataset()

metric_ddd = np.load(f'metric/{sys_config["case_name"]}/metric_ddd_test.npy', allow_pickle=True).item()

## TPR and FPR

In [ ]:
FP_DD_summary = np.zeros(len(dd_detector.quantile))
for input, output in test_dataloader_scaled:
    decoded, loss_recons = dd_detector.evaluate(input)
    for idx, quantile in enumerate(dd_detector.quantile):
        if loss_recons >= dd_detector.ae_threshold[idx]:
            FP_DD_summary[idx] += 1

FPR_DD_summary = FP_DD_summary / len(test_dataloader_scaled.dataset)


print(f'DD FPR: {FPR_DD_summary}')

In [ ]:
# Attack list
ang_no_list = [1,2,3]
mag_no = 0
ang_str_list = [0.2,0.3]
mag_str = 0

ROC_dict = {}                                                                                                                                                                                                           

for ang_no in ang_no_list:
    # Attack number
    for ang_str in ang_str_list:
        # Attack strength
        
        residual = []
        for _, (idx, input, v_est_pre, v_est_last) in tqdm(enumerate(test_dataloader_unscaled)):
            # Convert data format
            v_est_last = v_est_last.flatten()
            # Attack
            z_att_noise, v_att_est_last = case_class.gen_fdi_att_dd(z_noise=input, v_est_last=v_est_last, ang_no=ang_no, mag_no=mag_no, ang_str=ang_str, mag_str=mag_str) # z_attack_noise: (1,sample_length,feature_size)
            # Scale
            z_att_noise_scale = scaler_(z_att_noise)
            decoded,loss_recons = dd_detector.evaluate(z_att_noise_scale)

            ## unScale
            ## decoded, loss_recons = dd_detector.evaluate(z_att_noise)
            residual.append(loss_recons)
        
        TPR_DDD_summary = []
        # For each of the threshold, find the detection accuracy
        for i, quantile in enumerate(dd_detector.quantile):
            TPR_DDD = np.sum(residual >= dd_detector.ae_threshold[i])/len(residual)
            TPR_DDD_summary.append(TPR_DDD)
            
        ROC_dict[f'({ang_no},{ang_str})'] = TPR_DDD_summary

In [ ]:
plt.figure()
plt.plot(1-np.array(dd_detector.quantile), FPR_DD_summary, label = 'Test Set', color = cm.Dark2(0))
plt.plot(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), label = 'Valid Set', linestyle = 'dashed', color = cm.Dark2(0))
plt.vlines(x = 1-dd_detector.quantile[dd_detector.quantile_idx], ymin = 0, ymax = 1, colors = 'red', linestyles='--', label='Threshold')
plt.xlabel('FPR')
plt.ylabel('FPR')
plt.fill_between(1-np.array(dd_detector.quantile), 1-np.array(dd_detector.quantile), FPR_DD_summary, color = cm.Dark2(0), alpha = 0.1)
plt.legend()
plt.ylim([0.0,0.31])
plt.yticks(np.arange(0,0.31,0.05))
plt.xlim([0.0,0.2])
plt.grid()
plt.savefig(f'figures/{sys_config["case_name"]}/false_positive.pdf')
plt.show()

In [ ]:
# ROC curve
plt.figure()
for idx, key in enumerate(ROC_dict.keys()):
    plt.plot(FPR_DD_summary, ROC_dict[key], label = key, color = cm.Dark2(idx), linestyle = '-', marker = 'o')
    if idx == 3:
        break
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.grid()
plt.vlines(x = FPR_DD_summary[dd_detector.quantile_idx], ymin = 0, ymax = 1, colors = 'red', linestyles='--', label='Threshold')
plt.legend()
plt.ylim([0.45,1.0])
plt.xlim([0,0.30])
plt.savefig(f'figures/{sys_config["case_name"]}/roc.pdf')
plt.show()

print(f'The validation set FPR: {1-dd_detector.quantile[dd_detector.quantile_idx]}')
print(f'The false positive rate is: {np.sum(metric_ddd["FP_DDD"])/len(metric_ddd["FP_DDD"])}')

In [ ]:

TPR_DDD = []
attack_str = []
for key in metric_ddd['TP_DDD'].keys():
    TPR_DDD.append(np.sum(metric_ddd['TP_DDD'][key])/len(metric_ddd['TP_DDD'][key]))
    attack_str.append(metric_ddd['att_strength'][key])
print(TPR_DDD)

plt.plot(TPR_DDD)
plt.title('The TPR of DDD')
plt.grid()
plt.show()

# plt.figure()
# plt.boxplot(attack_str)
# plt.title('attack strength')
# plt.grid()
# plt.show()

print(f'minimum attack strength: {np.min(attack_str,1)}')

In [ ]:
## AUC，F1-score， ACC， FPR， MCC

from sklearn.metrics import auc, precision_score, recall_score, f1_score, accuracy_score, matthews_corrcoef
import numpy as np

try:
    metric_ddd = np.load(f'metric/{sys_config["case_name"]}/metric_ddd_test.npy', allow_pickle=True).item()
except FileNotFoundError:
    print("Warning: metric_ddd_test.npy not found. Please run the evaluation cells (e.g., Cell 4) first to generate metrics.")
    metric_ddd = None


if metric_ddd is not None:

    for i in range(0,4):
        try:
            model_key = list(ROC_dict.keys())[i]
            tpr_array = ROC_dict[model_key]
            fpr_array = FPR_DD_summary
            roc_auc = auc(fpr_array, tpr_array)
            print(f"AUC-ROC (Attack Case: {model_key}): {roc_auc:.4f}")
        except NameError:
            print("Warning: ROC_dict or FPR_DD_summary not defined. Cannot calculate AUC.")
        except IndexError:
            print("Warning: ROC_dict is empty. Cannot calculate AUC.")


    # --- 2. F1-score, ACC, FPR, MCC (基于选定的阈值) ---
    # 获取选定阈值下的 TP 结果 (所有攻击情况合并)
    all_tpr_results = []
    for key in metric_ddd['TP_DDD'].keys():
        all_tpr_results.extend(metric_ddd['TP_DDD'][key])

    # TP_DDD 记录的是每个攻击样本是否被检测到 (True=TP, False=FN)
    TP_FN_count = len(all_tpr_results) # TP + FN (所有攻击样本总数)
    TP_count = np.sum(all_tpr_results)
    FN_count = TP_FN_count - TP_count

    # FP_DDD 记录的是每个正常样本是否被错误地检测到 (True=FP, False=TN)
    FP_TN_count = len(metric_ddd['FP_DDD']) # FP + TN (所有正常样本总数)
    FP_count = np.sum(metric_ddd['FP_DDD'])
    TN_count = FP_TN_count - FP_count


    # Y_true: 攻击样本=1 (TP+FN), 正常样本=0 (FP+TN)
    Y_true = np.concatenate([np.ones(TP_FN_count), np.zeros(FP_TN_count)])
    
    # Y_pred: 模型检测到攻击=1 (TP+FP), 模型未检测到攻击=0 (FN+TN)
    all_fp_results = metric_ddd['FP_DDD']
    Y_pred = np.concatenate([np.array(all_tpr_results), np.array(all_fp_results)])


    # --- 指标计算 ---
    accuracy = accuracy_score(Y_true, Y_pred)
    precision = precision_score(Y_true, Y_pred, zero_division=0)
    recall = recall_score(Y_true, Y_pred, zero_division=0)  # Recall = TPR
    f1 = f1_score(Y_true, Y_pred, zero_division=0)
    
    # False positive rate: FPR = FP / (FP + TN).
    fpr = FP_count / FP_TN_count if FP_TN_count > 0 else 0.0
    
    # Matthews correlation coefficient (MCC).
    mcc = matthews_corrcoef(Y_true, Y_pred)

    # --- 结果打印 ---
    print("-" * 45)
    print(f"Metrics for selected Threshold (Quantile 0.92):")
    print("-" * 45)
    print(f"  [Sample Counts]")
    print(f"    Total Attacked Samples (TP+FN): {TP_FN_count}")
    print(f"    Total Normal Samples (FP+TN)  : {FP_TN_count}")
    print(f"    True Positives (TP)           : {TP_count}")
    print(f"    False Positives (FP)          : {FP_count}")
    print(f"    True Negatives (TN)           : {TN_count}") 
    print(f"    False Negatives (FN)          : {FN_count}")
    print(f"  [Evaluation Metrics]")
    print(f"    Accuracy (ACC)                : {accuracy:.4f}") 
    print(f"    Precision                     : {precision:.4f}")
    print(f"    Recall (TPR)                  : {recall:.4f}")
    print(f"    False Positive Rate (FPR)     : {fpr:.4f}")   # 打印 FPR
    print(f"    F1-Score                      : {f1:.4f}")
    print(f"    Matthews Corr Coeff (MCC)     : {mcc:.4f}")    # 打印 MCC
    print("-" * 45)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 计算混淆矩阵
# labels=[0, 1] 确保矩阵的顺序是：先 0(正常), 后 1(攻击)
cm = confusion_matrix(Y_true, Y_pred, labels=[0, 1])

# 提取具体数值
tn, fp, fn, tp = cm.ravel()

print("-" * 30)
print("混淆矩阵 (Confusion Matrix):")
print(f" [TN (真阴性): {tn}]  [FP (假阳性): {fp}]")
print(f" [FN (假阴性): {fn}]  [TP (真阳性): {tp}]")
print("-" * 30)

# 2. 绘制热力图 (更直观)
plt.figure(figsize=(6, 5))
labels = ['Normal (0)', 'Attack (1)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels, yticklabels=labels, annot_kws={"size": 16})

plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title('Confusion Matrix', fontsize=14)
plt.show()

# --- 3. Threshold diagnostics / 阈值诊断 ---
if tn == 0 and fp > 0:
    print("\nTN = 0：当前阈值下，全部正常样本被分类为攻击。")
    print("本次评估 FPR = 1.0。")
    print("阈值与比较方向见前面的评估设置。")

if tp == 0 and fn > 0:
    print("\nTP = 0：当前阈值下，全部攻击样本被分类为正常。")
    print("本次评估 Recall = 0.0。")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import auc

# 1. 获取数据和样本量
# Count normal and attack windows from their evaluation datasets.
num_neg = len(test_dataloader_scaled.dataset)  
num_pos = len(test_dataloader_unscaled.dataset) 

# 选择一个攻击场景 (例如第一个 key)
model_key = list(ROC_dict.keys())[0] 
tpr_array = np.array(ROC_dict[model_key])
fpr_array = np.array(FPR_DD_summary)

# 2. 排序 (sklearn.auc 要求 x 轴单调)
# Sort points by ascending FPR for numerical integration.
sort_idx = np.argsort(fpr_array)
fpr_sorted = fpr_array[sort_idx]
tpr_sorted = tpr_array[sort_idx]

# --- 计算 AUC ---
roc_auc = auc(fpr_sorted, tpr_sorted)

# --- 计算 F1-Score ---
# Precision = TP / (TP + FP)
# TP = TPR * num_pos
# FP = FPR * num_neg
tp_count = tpr_sorted * num_pos
fp_count = fpr_sorted * num_neg
epsilon = 1e-8 # 防止除以 0

precision_array = tp_count / (tp_count + fp_count + epsilon)
recall_array = tpr_sorted  # Recall 等于 TPR

f1_array = 2 * (precision_array * recall_array) / (precision_array + recall_array + epsilon)

# 找到最大 F1 值及其对应的阈值索引
max_f1_idx = np.argmax(f1_array)
max_f1 = f1_array[max_f1_idx]

# 注意：sort_idx[max_f1_idx] 才是原始 quantile 数组中对应的索引
original_best_idx = sort_idx[max_f1_idx]
best_threshold = dd_detector.ae_threshold[original_best_idx]
best_quantile = dd_detector.quantile[original_best_idx]

# 3. 打印结果
print(f"Model Scenario: {model_key}")
print(f"AUC-ROC: {roc_auc:.4f}")
print(f"Max F1-Score: {max_f1:.4f}")
print(f"Best Threshold (from F1): {best_threshold:.4f} (Quantile: {best_quantile})")

# 4. (可选) 绘制 F1 曲线
plt.figure(figsize=(10, 4))

# Plot ROC
plt.subplot(1, 2, 1)
plt.plot(fpr_sorted, tpr_sorted, label=f'AUC = {roc_auc:.2f}', marker='o')
plt.title('ROC Curve')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.grid(True)
plt.legend()

# Plot F1 vs FPR (或者可以画 F1 vs Quantile)
plt.subplot(1, 2, 2)
plt.plot(fpr_sorted, f1_array, label='F1-Score', color='orange', marker='x')
plt.scatter(fpr_sorted[max_f1_idx], max_f1, color='red', s=100, label='Max F1')
plt.title(f'F1-Score (Max: {max_f1:.2f})')
plt.xlabel('FPR')
plt.ylabel('F1 Score')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Validation ROC-AUC based on continuous reconstruction errors
from sklearn.metrics import roc_auc_score

# Normal validation samples (negative class)
valid_normal_scores = []
for input_scaled, _ in tqdm(valid_dataloader_scaled, desc='Validation normal scores'):
    _, loss_recons = dd_detector.evaluate(input_scaled)
    valid_normal_scores.append(loss_recons)
valid_normal_scores = np.asarray(valid_normal_scores)

# Attacked validation samples (positive class)
validation_auc = {}
all_attack_scores = []
for ang_no in ang_no_list:
    for ang_str in ang_str_list:
        attack_scores = []
        scenario = f'({ang_no},{ang_str})'
        for _, input_unscaled, _, v_est_last in tqdm(
            valid_dataloader_unscaled, desc=f'Validation attack {scenario}'
        ):
            z_att_noise, _ = case_class.gen_fdi_att_dd(
                z_noise=input_unscaled,
                v_est_last=v_est_last.flatten(),
                ang_no=ang_no,
                mag_no=mag_no,
                ang_str=ang_str,
                mag_str=mag_str,
            )
            _, loss_recons = dd_detector.evaluate(scaler_(z_att_noise))
            attack_scores.append(loss_recons)

        attack_scores = np.asarray(attack_scores)
        y_true = np.concatenate([
            np.zeros(valid_normal_scores.size),
            np.ones(attack_scores.size),
        ])
        y_score = np.concatenate([valid_normal_scores, attack_scores])
        validation_auc[scenario] = roc_auc_score(y_true, y_score)
        all_attack_scores.append(attack_scores)
        print(f'Validation AUC-ROC {scenario}: {validation_auc[scenario]:.4f}')

all_attack_scores = np.concatenate(all_attack_scores)
overall_y_true = np.concatenate([
    np.zeros(valid_normal_scores.size),
    np.ones(all_attack_scores.size),
])
overall_y_score = np.concatenate([valid_normal_scores, all_attack_scores])
validation_auc_macro = np.mean(list(validation_auc.values()))
validation_auc_overall = roc_auc_score(overall_y_true, overall_y_score)

print('-' * 45)
print(f'Validation AUC-ROC (macro):   {validation_auc_macro:.10f}')
print(f'Validation AUC-ROC (overall): {validation_auc_overall:.10f}')
